# Free AI avatar studio: fast lip sync (EchoMimicV3-Flash on Kaggle's free T4 x2)
Part of [free-ai-avatar-studio](https://github.com/sahilempire/free-ai-avatar-studio). Turns a kit made by `studio kit`
(scene images + voice-over chunks + manifest) into one lip-synced clip per chunk, then zips them.

1. Upload `work/kit.zip` as a **private** Kaggle dataset, then **Add Input** here.
2. **Settings:** Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified Kaggle account).
3. **Run All** (or *Save Version → Save & Run All* to run in the background). About 1.5 h for 60 s of speech.
4. Download `clips.zip` from the Output panel, unzip into your project's `work/clips/`, then `studio post`.

Re-render a few clips: set `ONLY` (and `SEEDS` for a different take) and run again.

In [ ]:
# ---- Settings ----
SAMPLE_SIZE = 640     # ~464x848 output (upscaled locally). 768 is sharper but ~1.8x the GPU time.
STEPS = 8             # Flash is tuned for 8 steps
SEED = 43
SEEDS = {}            # per-clip seed for re-renders, e.g. {"c03": 44}
ONLY = None           # e.g. ["c00", "c01"] to test or re-render some clips
NEG = 'blurry, low quality, deformed face, distorted face, changing face, asymmetric eyes, distorted hands, extra fingers, fused fingers, melting phone, smeared objects, flicker, warping background, text, subtitles, watermark, logo'
DEFAULT_PROMPT = 'A person talks directly to the camera with natural lip movement, natural blinking, expressive eyebrows and subtle head movement. Realistic skin, natural light, hands steady.'

import os, glob, json, math, shutil, subprocess, threading, time, base64, re, torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"
print(torch.__version__, torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "Add your kit as input: upload work/kit.zip (from `studio kit`) as a Kaggle dataset, then Add Input"
KIT = os.path.dirname(hits[0]); print("kit:", KIT)
def find(rel):  # Kaggle can flatten a dataset's folders, so fall back to looking files up by name
    hits = glob.glob(f"{KIT}/{rel}") or glob.glob(f"/kaggle/input/**/{os.path.basename(rel)}", recursive=True)
    assert hits, f"not in the dataset: {rel}"; return hits[0]
LOGS = "/kaggle/working/logs"; os.makedirs(LOGS, exist_ok=True); T0 = time.time()
def sh(cmd, log, cwd=None, env=None):
    t = time.time()
    with open(f"{LOGS}/{log}.log", "w") as f:
        r = subprocess.run(cmd, shell=True, cwd=cwd, env=env, stdout=f, stderr=subprocess.STDOUT)
    print(f"[{(time.time()-T0)/60:6.1f}m] {log}: exit {r.returncode} in {(time.time()-t)/60:.1f} min", flush=True)
    assert r.returncode == 0, f"{log} failed:\n" + "".join(open(f"{LOGS}/{log}.log").readlines()[-25:])
MEMPEAK, STOP = [0.0], threading.Event()
def mem_watch():  # system RAM log (two renderers at once can get OOM-killed)
    with open(f"{LOGS}/mem.log", "w") as f:
        while not STOP.is_set():
            m = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo")}
            used = (m["MemTotal"] - m["MemAvailable"]) / 2**20; MEMPEAK[0] = max(MEMPEAK[0], used)
            f.write(f"{(time.time()-T0)/60:6.1f}m used {used:5.1f} GiB\n"); f.flush(); STOP.wait(10)
threading.Thread(target=mem_watch, daemon=True).start()

DTYPE = "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"   # T4: native fp16
OUT, MODELS, REPO = "/kaggle/working/clips", "/tmp/em3_models", "/tmp/echomimic_v3"
os.makedirs(OUT, exist_ok=True); os.makedirs(MODELS, exist_ok=True)

In [ ]:
# ---- Code + dependencies (Kaggle's torch is kept) ----
if not os.path.exists(REPO):
    sh(f"git clone -q https://github.com/antgroup/echomimic_v3.git {REPO} && cd {REPO} && git checkout -q 7e89489ca51c0d008fc1963ec6c03fc5bd0b9397", "clone")
sh('pip install -q "diffusers==0.33.1" "transformers==4.51.3" "accelerate==1.6.0" einops safetensors timm tomesd torchdiffeq '
   'torchsde decord omegaconf sentencepiece ftfy func_timeout "moviepy==2.2.1" librosa pyloudnorm "imageio[ffmpeg]" av soundfile hf_transfer', "deps")

In [ ]:
# ---- Weights (~24 GB, kept in /tmp) ----
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download, hf_hub_download
BASE = f"{MODELS}/Wan2.1-Fun-V1.1-1.3B-InP"     # folder name matters for TeaCache
snapshot_download("alibaba-pai/Wan2.1-Fun-V1.1-1.3B-InP", local_dir=BASE, allow_patterns=["*.json", "*.pth", "*.safetensors", "google/*", "xlm-roberta-large/*"])
W2V = f"{MODELS}/chinese-wav2vec2-base"
snapshot_download("TencentGameMate/chinese-wav2vec2-base", local_dir=W2V, allow_patterns=["config.json", "preprocessor_config.json", "pytorch_model.bin"])
hf_hub_download("BadToBest/EchoMimicV3", "echomimicv3-flash-pro/diffusion_pytorch_model.safetensors", local_dir=f"{MODELS}/flash")
FLASH = f"{MODELS}/flash/echomimicv3-flash-pro/diffusion_pytorch_model.safetensors"
print("weights ready:", os.path.exists(FLASH))

In [ ]:
# ---- Patch upstream into a batch renderer; pad each chunk so the clip covers all of its speech; split across GPUs ----
for name, blob in (("patch_echomimic.py", "IiIiVHVybnMgRWNob01pbWljVjMncyBpbmZlcl9mbGFzaC5weSAoY29tbWl0IDdlODk0ODkpIGludG8gYmF0Y2gucHksIGEgYmF0Y2ggcmVuZGVyZXIgZm9yIGEgc3R1ZGlvIGtpdDoKLSByZW5kZXJzIGV2ZXJ5IGpvYiBpbiBhIG1hbmlmZXN0IHdpdGggdGhlIG1vZGVscyBsb2FkZWQgb25jZSAoc2tpcHMgY2xpcHMgdGhhdCBhbHJlYWR5IGV4aXN0KQotIGxvYWRzIHRoZSBGbGFzaCB0cmFuc2Zvcm1lciB3aXRoIGFzc2lnbj1UcnVlOiB0aGUgYmFzZSBXYW4gY2hlY2twb2ludCBoYXMgbm8gYXVkaW8gbGF5ZXJzLCBzbyB0aGV5IHN0YXJ0IG9uCiAgdGhlIG1ldGEgZGV2aWNlIGFuZCBhIHBsYWluIGxvYWRfc3RhdGVfZGljdCBsZWF2ZXMgdGhlbSBlbXB0eSAoIkNhbm5vdCBjb3B5IG91dCBvZiBtZXRhIHRlbnNvciIpCi0gZnJlZXMgdGhlIDMuNyBHQiBzdGF0ZV9kaWN0IHVwc3RyZWFtIGtlZXBzIGFsaXZlIGZvciB0aGUgd2hvbGUgcnVuCi0gbmV2ZXIgbG9hZHMgdGhlIDExIEdCIFQ1OiBwcm9tcHRzIGNvbWUgcHJlLWVuY29kZWQgKGVtYmVkX3Byb21wdHMucHkpIHRocm91Z2ggZW5jb2RlX3Byb21wdAotIG1vZGVsIENQVSBvZmZsb2FkIHNvIGl0IGZpdHMgYSAxNiBHQiBUNDsgcGVyLWpvYiBzZWVkClVzYWdlOiBweXRob24gcGF0Y2hfZWNob21pbWljLnB5IDxyZXBvPgoiIiIKaW1wb3J0IHJlLCBzeXMKCnJlcG8gPSBzeXMuYXJndlsxXQpzcmMgPSBvcGVuKGYie3JlcG99L2luZmVyX2ZsYXNoLnB5IiwgZW5jb2Rpbmc9InV0Zi04IikucmVhZCgpCgoKZGVmIHN3YXAob2xkLCBuZXcsIGNvdW50PTEpOgogICAgZ2xvYmFsIHNyYwogICAgbiA9IHNyYy5jb3VudChvbGQpCiAgICBhc3NlcnQgbiA9PSBjb3VudCwgZiJleHBlY3RlZCB7Y291bnR9eCB7b2xkWzo2MF0hcn0sIGZvdW5kIHtufSAodXBzdHJlYW0gY2hhbmdlZD8gcGluIHRoZSBjb21taXQpIgogICAgc3JjID0gc3JjLnJlcGxhY2Uob2xkLCBuZXcpCgoKIyBhcmd1bWVudHM6IHBlci1qb2IgdmFsdWVzIGNvbWUgZnJvbSB0aGUgbWFuaWZlc3QKZm9yIGFyZyBpbiAoImltYWdlX3BhdGgiLCAiYXVkaW9fcGF0aCIsICJwcm9tcHQiKToKICAgIHN3YXAoZidwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXthcmd9IiwgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUnLCBmJ3BhcnNlci5hZGRfYXJndW1lbnQoIi0te2FyZ30iLCB0eXBlPXN0ciwgZGVmYXVsdD0iIicpCnN3YXAoIiAgICByZXR1cm4gcGFyc2VyLnBhcnNlX2FyZ3MoKSIsICIgICAgcGFyc2VyLmFkZF9hcmd1bWVudChcIi0tbWFuaWZlc3RcIiwgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUpXG4iCiAgICAgIiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KFwiLS1raXRfZGlyXCIsIHR5cGU9c3RyLCByZXF1aXJlZD1UcnVlKVxuIgogICAgICIgICAgcGFyc2VyLmFkZF9hcmd1bWVudChcIi0tZW1iZWRzXCIsIHR5cGU9c3RyLCByZXF1aXJlZD1UcnVlKVxuIgogICAgICIgICAgcmV0dXJuIHBhcnNlci5wYXJzZV9hcmdzKCkiKQojIEZsYXNoIHdlaWdodHMgaW50byB0aGUgbWV0YS1pbml0aWFsaXNlZCBhdWRpbyBsYXllcnMsIHRoZW4gZHJvcCB0aGUgZHVwbGljYXRlCnN3YXAoIiAgICAgICAgbSwgdSA9IHRyYW5zZm9ybWVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZV9kaWN0LCBzdHJpY3Q9RmFsc2UpIiwKICAgICAiICAgICAgICBzdGF0ZV9kaWN0ID0ge2s6IHYudG8od2VpZ2h0X2R0eXBlKSBmb3IgaywgdiBpbiBzdGF0ZV9kaWN0Lml0ZW1zKCl9XG4iCiAgICAgIiAgICAgICAgbSwgdSA9IHRyYW5zZm9ybWVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZV9kaWN0LCBzdHJpY3Q9RmFsc2UsIGFzc2lnbj1UcnVlKVxuIgogICAgICIgICAgICAgIGFzc2VydCBub3QgW24gZm9yIG4sIHAgaW4gdHJhbnNmb3JtZXIubmFtZWRfcGFyYW1ldGVycygpIGlmIHAuaXNfbWV0YV0sIFwid2VpZ2h0cyBtaXNzaW5nIGZyb20gdGhlIGNoZWNrcG9pbnRcIlxuIgogICAgICIgICAgICAgIGltcG9ydCBnYzsgZGVsIHN0YXRlX2RpY3Q7IGdjLmNvbGxlY3QoKSIpCiMgbm8gVDU6IGVtYmVkZGluZ3MgYXJlIGxvYWRlZCBpbnN0ZWFkCnQ1ID0gcmUuc2VhcmNoKHIiICAgICMgR2V0IFRleHQgZW5jb2RlclxuICAgIHRleHRfZW5jb2RlciA9IFdhblQ1RW5jb2Rlck1vZGVsXC5mcm9tX3ByZXRyYWluZWRcKC4qP1xuICAgIHRleHRfZW5jb2RlciA9IHRleHRfZW5jb2RlclwuZXZhbFwoXCkiLCBzcmMsIHJlLlMpCmFzc2VydCB0NSwgIlQ1IGJsb2NrIG5vdCBmb3VuZCIKc3JjID0gc3JjLnJlcGxhY2UodDUuZ3JvdXAoMCksICIgICAgdGV4dF9lbmNvZGVyID0gTm9uZSAgIyBwcm9tcHRzIGFyZSBwcmUtZW5jb2RlZFxuICAgIEVNQiA9IHRvcmNoLmxvYWQoYXJncy5lbWJlZHMpIikKIyBvZmZsb2FkIGluc3RlYWQgb2YgbW92aW5nIGV2ZXJ5dGhpbmcgdG8gdGhlIEdQVSAodXBzdHJlYW0gY2FsbHMgcGlwZWxpbmUudG8gdHdpY2UpCmFzc2VydCBzcmMuY291bnQoIiAgICBwaXBlbGluZS50byhkZXZpY2U9ZGV2aWNlKSIpID09IDIsICJleHBlY3RlZCB0d28gcGlwZWxpbmUudG8oKSBjYWxscyIKc3JjID0gc3JjLnJlcGxhY2UoIiAgICBwaXBlbGluZS50byhkZXZpY2U9ZGV2aWNlKSIsICIgICAgcGlwZWxpbmUuZW5hYmxlX21vZGVsX2NwdV9vZmZsb2FkKCkiLCAxKQpzcmMgPSBzcmMucmVwbGFjZSgiXG4gICAgcGlwZWxpbmUudG8oZGV2aWNlPWRldmljZSkiLCAiIiwgMSkKc3dhcCgiICAgIGNvZWZmaWNpZW50cyA9IGdldF90ZWFjYWNoZV9jb2VmZmljaWVudHMobW9kZWxfbmFtZSkgaWYgZW5hYmxlX3RlYWNhY2hlIGVsc2UgTm9uZSIsCiAgICAgIiAgICBkZWYgX2VuY29kZV9wcm9tcHQocHJvbXB0LCBuZWdhdGl2ZV9wcm9tcHQ9Tm9uZSwgZG9fY2xhc3NpZmllcl9mcmVlX2d1aWRhbmNlPVRydWUsICphLCBkZXZpY2U9Tm9uZSwgZHR5cGU9Tm9uZSwgKiprKTpcbiIKICAgICAiICAgICAgICB0byA9IGRpY3QoZGV2aWNlPWRldmljZSBvciBcImN1ZGFcIiwgZHR5cGU9ZHR5cGUgb3Igd2VpZ2h0X2R0eXBlKVxuIgogICAgICIgICAgICAgIHJldHVybiBbRU1CW3Byb21wdF0udG8oKip0byldLCAoW0VNQltcIl9fbmVnX19cIl0udG8oKip0byldIGlmIGRvX2NsYXNzaWZpZXJfZnJlZV9ndWlkYW5jZSBlbHNlIE5vbmUpXG4iCiAgICAgIiAgICBwaXBlbGluZS5lbmNvZGVfcHJvbXB0ID0gX2VuY29kZV9wcm9tcHRcblxuIgogICAgICIgICAgY29lZmZpY2llbnRzID0gZ2V0X3RlYWNhY2hlX2NvZWZmaWNpZW50cyhtb2RlbF9uYW1lKSBpZiBlbmFibGVfdGVhY2FjaGUgZWxzZSBOb25lIikKCiMgd3JhcCB0aGUgcGVyLWNsaXAgZ2VuZXJhdGlvbiAodGhlIGxhc3QgdG9wLWxldmVsIGB3aXRoIHRvcmNoLm5vX2dyYWQoKTpgIGluIG1haW4pIGluIGEgbG9vcCBvdmVyIHRoZSBtYW5pZmVzdApsaW5lcyA9IHNyYy5zcGxpdCgiXG4iKQpzdGFydCA9IG1heChpIGZvciBpLCBsIGluIGVudW1lcmF0ZShsaW5lcykgaWYgbCA9PSAiICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOiIpCnN0b3AgPSBuZXh0KGkgZm9yIGksIGwgaW4gZW51bWVyYXRlKGxpbmVzKSBpZiBsLnN0YXJ0c3dpdGgoImlmIF9fbmFtZV9fIikpCmJsb2NrID0gWygiICAgICIgKyBsIGlmIGwuc3RyaXAoKSBlbHNlIGwpIGZvciBsIGluIGxpbmVzW3N0YXJ0OnN0b3BdXQp3aGlsZSBibG9jayBhbmQgbm90IGJsb2NrWy0xXS5zdHJpcCgpOiBibG9jay5wb3AoKQpsb29wID0gWyIgICAgaW1wb3J0IGpzb24iLAogICAgICAgICIgICAgZm9yIGpvYiBpbiBqc29uLmxvYWQob3BlbihhcmdzLm1hbmlmZXN0KSk6IiwKICAgICAgICAiICAgICAgICBpbWFnZV9wYXRoID0gb3MucGF0aC5qb2luKGFyZ3Mua2l0X2Rpciwgam9iW1wiaW1hZ2VcIl0pIiwKICAgICAgICAiICAgICAgICBhdWRpb19wYXRoID0gb3MucGF0aC5qb2luKGFyZ3Mua2l0X2Rpciwgam9iW1wiYXVkaW9cIl0pIiwKICAgICAgICAiICAgICAgICBwcm9tcHQsIHZpZGVvX2xlbmd0aCA9IGpvYltcInByb21wdFwiXSwgam9iW1wiZnJhbWVzXCJdIiwKICAgICAgICAiICAgICAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPVwiY3VkYVwiKS5tYW51YWxfc2VlZChqb2IuZ2V0KFwic2VlZFwiLCBzZWVkKSkiXSArIGJsb2NrICsgWyIiLCAiIl0Kc3JjID0gIlxuIi5qb2luKGxpbmVzWzpzdGFydF0gKyBsb29wICsgbGluZXNbc3RvcDpdKQpzd2FwKCJpbWFnZV9uYW1lID0gb3MucGF0aC5iYXNlbmFtZShpbWFnZV9wYXRoKS5zcGxpdCgnLicpWzBdIiwgImltYWdlX25hbWUgPSBqb2JbJ2lkJ10iKQpza2lwID0gcmUuc2VhcmNoKHIiKHByaW50XChmXCLij63vuI8gIFNraXA6IFx7b3V0cHV0X3ZpZGVvX3BhdGhcfSBhbHJlYWR5IGV4aXN0c1wuXCJcKVxuXHMrKXJldHVybiIsIHNyYykKYXNzZXJ0IHNraXAsICJza2lwLWV4aXN0aW5nIGJsb2NrIG5vdCBmb3VuZCIKc3JjID0gc3JjLnJlcGxhY2Uoc2tpcC5ncm91cCgwKSwgc2tpcC5ncm91cCgxKSArICJjb250aW51ZSIpCm9wZW4oZiJ7cmVwb30vYmF0Y2gucHkiLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpLndyaXRlKHNyYykKCiMgdGhlIHBpcGVsaW5lIHJlYWRzIHRoZSBUNSBkdHlwZSBvbiBldmVyeSBjYWxsCnAgPSBmIntyZXBvfS9zcmMvcGlwZWxpbmVfd2FuX2Z1bl9pbnBhaW50X2F1ZGlvXzI1MTIucHkiCnBzID0gb3BlbihwLCBlbmNvZGluZz0idXRmLTgiKS5yZWFkKCkKaWYgIndlaWdodF9kdHlwZSA9IHNlbGYudGV4dF9lbmNvZGVyLmR0eXBlIiBpbiBwczoKICAgIG9wZW4ocCwgInciLCBlbmNvZGluZz0idXRmLTgiKS53cml0ZShwcy5yZXBsYWNlKCJ3ZWlnaHRfZHR5cGUgPSBzZWxmLnRleHRfZW5jb2Rlci5kdHlwZSIsICJ3ZWlnaHRfZHR5cGUgPSBzZWxmLnRyYW5zZm9ybWVyLmR0eXBlIikpCnByaW50KCJiYXRjaC5weSByZWFkeSIpCg=="), ("embed_prompts.py", "IyBFbmNvZGVzIGV2ZXJ5IGRpc3RpbmN0IHByb21wdCAoYW5kIHRoZSBuZWdhdGl2ZSBwcm9tcHQpIHdpdGggdW1UNSBvbmNlLCBpbiBiZjE2IChpdHMgbmF0aXZlIGR0eXBlKSwKIyB1c2luZyB0aGUgcGlwZWxpbmUncyBvd24gX2dldF90NV9wcm9tcHRfZW1iZWRzIHNvIHRoZSByZXN1bHQgbWF0Y2hlcyB3aGF0IHRoZSBwaXBlbGluZSB3b3VsZCBtYWtlLgojIFVzYWdlIChmcm9tIHRoZSByZXBvIGZvbGRlcik6IHB5dGhvbiBlbWJlZF9wcm9tcHRzLnB5IDxtb2RlbF9uYW1lPiA8Y29uZmlnLnlhbWw+IDxqb2JzLmpzb24+IDxuZWdhdGl2ZSBwcm9tcHQ+IDxvdXQucHQ+CmltcG9ydCBvcywgc3lzLCBqc29uLCB0eXBlcywgdG9yY2gKZnJvbSBvbWVnYWNvbmYgaW1wb3J0IE9tZWdhQ29uZgpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Rva2VuaXplcgpmcm9tIHNyYy53YW5fdGV4dF9lbmNvZGVyIGltcG9ydCBXYW5UNUVuY29kZXJNb2RlbApmcm9tIHNyYy5waXBlbGluZV93YW5fZnVuX2lucGFpbnRfYXVkaW9fMjUxMiBpbXBvcnQgV2FuRnVuSW5wYWludEF1ZGlvUGlwZWxpbmUKCm1vZGVsX25hbWUsIGNvbmZpZ19wYXRoLCBqb2JzX3BhdGgsIG5lZ2F0aXZlLCBvdXQgPSBzeXMuYXJndlsxOjZdCmNvbmZpZyA9IE9tZWdhQ29uZi5sb2FkKGNvbmZpZ19wYXRoKQprdyA9IGNvbmZpZ1sidGV4dF9lbmNvZGVyX2t3YXJncyJdCnRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG9zLnBhdGguam9pbihtb2RlbF9uYW1lLCBrdy5nZXQoInRva2VuaXplcl9zdWJwYXRoIiwgInRva2VuaXplciIpKSkKdGV4dF9lbmNvZGVyID0gV2FuVDVFbmNvZGVyTW9kZWwuZnJvbV9wcmV0cmFpbmVkKAogICAgb3MucGF0aC5qb2luKG1vZGVsX25hbWUsIGt3LmdldCgidGV4dF9lbmNvZGVyX3N1YnBhdGgiLCAidGV4dF9lbmNvZGVyIikpLAogICAgYWRkaXRpb25hbF9rd2FyZ3M9T21lZ2FDb25mLnRvX2NvbnRhaW5lcihrdyksIGxvd19jcHVfbWVtX3VzYWdlPVRydWUsIHRvcmNoX2R0eXBlPXRvcmNoLmJmbG9hdDE2LAopLmV2YWwoKS50bygiY3VkYSIpCmZha2VfcGlwZSA9IHR5cGVzLlNpbXBsZU5hbWVzcGFjZSh0b2tlbml6ZXI9dG9rZW5pemVyLCB0ZXh0X2VuY29kZXI9dGV4dF9lbmNvZGVyLCBfZXhlY3V0aW9uX2RldmljZT10b3JjaC5kZXZpY2UoImN1ZGEiKSkKcHJvbXB0cyA9IHNvcnRlZCh7alsicHJvbXB0Il0gZm9yIGogaW4ganNvbi5sb2FkKG9wZW4oam9ic19wYXRoKSl9KQplbWIgPSB7fQp3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgIGZvciBrZXksIHRleHQgaW4gWyhwLCBwKSBmb3IgcCBpbiBwcm9tcHRzXSArIFsoIl9fbmVnX18iLCBuZWdhdGl2ZSldOgogICAgICAgIGUgPSBXYW5GdW5JbnBhaW50QXVkaW9QaXBlbGluZS5fZ2V0X3Q1X3Byb21wdF9lbWJlZHMoZmFrZV9waXBlLCB0ZXh0LCBkZXZpY2U9dG9yY2guZGV2aWNlKCJjdWRhIiksIGR0eXBlPXRvcmNoLmZsb2F0MTYpWzBdCiAgICAgICAgYXNzZXJ0IHRvcmNoLmlzZmluaXRlKGUpLmFsbCgpLCBmIm5vbi1maW5pdGUgZW1iZWRkaW5nIGZvciB7a2V5Wzo0MF0hcn0iCiAgICAgICAgZW1iW2tleV0gPSBlLmNwdSgpCiAgICAgICAgcHJpbnQoZiJlbmNvZGVkIHt0dXBsZShlLnNoYXBlKX0gIHtrZXlbOjYwXX0iKQp0b3JjaC5zYXZlKGVtYiwgb3V0KQpwcmludCgic2F2ZWQiLCBsZW4oZW1iKSwgImVtYmVkZGluZ3MgdG8iLCBvdXQpCg==")):
    open(f"{REPO}/{name}", "w").write(base64.b64decode(blob).decode())
sh(f"cd {REPO} && git checkout -q -- . && python patch_echomimic.py {REPO}", "patch")
import numpy as np, soundfile as sf
RUN_KIT = "/tmp/kit_run"; os.makedirs(f"{RUN_KIT}/audio", exist_ok=True); os.makedirs(f"{RUN_KIT}/images", exist_ok=True)
jobs = json.load(open(f"{KIT}/manifest.json"))
if ONLY: jobs = [j for j in jobs if j["id"] in ONLY]
for j in jobs:
    if not os.path.exists(f"{RUN_KIT}/{j['image']}"): os.symlink(find(j["image"]), f"{RUN_KIT}/{j['image']}")
    y, sr = sf.read(find(j["audio"]))
    need = math.ceil(len(y) / sr * 25); n = (need + 2) // 4 * 4 + 1        # upstream rounds frames DOWN twice
    pad = max(0, round((n + 0.5) / 25 * sr) - len(y))
    sf.write(f"{RUN_KIT}/{j['audio']}", np.concatenate([y, np.zeros((pad,) + y.shape[1:], y.dtype)]), sr)
    j["frames"] = max(j["frames"], n); j["prompt"] = j.get("prompt") or DEFAULT_PROMPT
    if j["id"] in SEEDS: j["seed"] = SEEDS[j["id"]]
json.dump(jobs, open("/tmp/jobs.json", "w"))
N_GPU = torch.cuda.device_count(); bins = [[] for _ in range(N_GPU)]
for j in sorted(jobs, key=lambda j: -j["frames"]): min(bins, key=lambda b: sum(x["frames"] for x in b)).append(j)
bins = [b for b in bins if b]; N_GPU = len(bins)
for g, b in enumerate(bins):
    json.dump(b, open(f"/tmp/jobs_gpu{g}.json", "w")); print(f"gpu{g}:", [x["id"] for x in b], sum(x["frames"] for x in b), "frames")

In [ ]:
# ---- Render: encode prompts once (no 11 GB T5 per renderer), one renderer per GPU, restart if killed ----
EMB = "/tmp/emb.pt"
if not os.path.exists(EMB):
    r = subprocess.run(["python", "embed_prompts.py", BASE, "config/config.yaml", "/tmp/jobs.json", NEG, EMB], cwd=REPO); assert r.returncode == 0
common = (f'--kit_dir {RUN_KIT} --embeds {EMB} --config_path config/config.yaml --model_name "{BASE}" --transformer_path "{FLASH}" '
          f'--wav2vec_model_dir "{W2V}" --save_path "{OUT}" --num_inference_steps {STEPS} --sampler_name Flow_Unipc --guidance_scale 6.0 '
          f'--audio_guidance_scale 3.0 --audio_scale 1.0 --neg_scale 1.0 --neg_steps 0 --seed {SEED} --enable_teacache --teacache_threshold 0.1 '
          f'--num_skip_start_steps 5 --weight_dtype {DTYPE} --sample_size {SAMPLE_SIZE} {SAMPLE_SIZE} --fps 25 --shift 5.0')
rc = {}
def run(gpu):
    cmd = f"cd {REPO} && CUDA_VISIBLE_DEVICES={gpu} python -u batch.py --manifest /tmp/jobs_gpu{gpu}.json {common}"
    with open(f"{LOGS}/gpu{gpu}.log", "a") as log:
        for attempt in range(3):
            p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:
                stamp = f"[gpu{gpu} {(time.time()-T0)/60:5.1f}m] "; log.write(stamp + line); log.flush()
                if any(k in line for k in ("Saved output", "Traceback", "Error", "Killed")): print(stamp + line, end="", flush=True)
            rc[gpu] = p.wait()
            if rc[gpu] == 0: break
            print(f"[gpu{gpu}] exited {rc[gpu]}, restarting (finished clips are skipped)", flush=True)
ts = [threading.Thread(target=run, args=(g,)) for g in range(N_GPU)]
for t in ts: t.start()
for t in ts: t.join()
STOP.set(); print("exit codes", rc, f"| peak RAM {MEMPEAK[0]:.1f} GiB | {(time.time()-T0)/60:.1f} min")

In [ ]:
# ---- Pack ----
shutil.make_archive("/kaggle/working/clips", "zip", OUT)
missing = [j["id"] for j in jobs if not os.path.exists(f"{OUT}/{j['id']}_output.mp4")]
print(len(jobs) - len(missing), "of", len(jobs), "clips | download /kaggle/working/clips.zip")
assert not missing, f"clips missing: {missing} (see /kaggle/working/logs); run again to finish them"